# TN0 — dựng lại kết quả MobiVital, rồi chứng minh pipeline của mình tương đương

Chạy một mạch trên Colab, khoảng 90 phút. Không cần notebook nào chạy trước.

## Nguyên tắc

Phần MobiVital chạy **đúng lệnh trong README của họ**, từ trong thư mục repo của
họ, không qua lớp bọc nào:

```bash
python dataset_preparation/prep_breath_final.py
python -m training.autoreg_training
python -m inference.mobivital_gen
python -m inference.evaluate -m YOUR_METHOD.txt
```

Chỉ xen thêm lệnh `mv` đổi tên bảng kết quả giữa hai lần chạy — vì
`mobivital_gen.py` dòng 121 luôn ghi ra đúng một tên
`{mode}_mobivital_pre_invert_{corr}.txt`, chạy hai lần cùng `corr` là đè mất lần
trước. Chính README của họ viết `-m YOUR_METHOD.txt`, tức là họ tính sẵn việc
người dùng tự đổi tên.

Cuối notebook in `git status` của repo MobiVital — phải trống.

## Một bản CSV duy nhất

```
external/mobivital/dataset/mobivital/tripod/   1874 CSV, giải nén thẳng vào đây
        |
        +--> prep_breath_final.py cua HO   -> data_final/*.npy
        |
        +--> scripts/make_npz.py cua MINH  -> data/processed/by_user/*.npz
                        |
              muc 2 doi chieu: phai bang nhau
```

`data/` chỉ chứa thứ pipeline của mình sinh ra. Không giữ bản CSV thứ hai ở đâu.

## Ba bậc, làm hai lần

```
        viec                            MobiVital              cua minh
   a    cham bang commit san            evaluate.py            scoring.score_from_txt
   b    checkpoint co san -> chon kenh  mobivital_gen.py       scoring.score_all
   c    train lai tu dau                autoreg_training.py    training.train
```

Mục 3–5 chạy cột giữa, mục 7 chạy cột phải, mục 8 đặt hai cột cạnh nhau.

Cần pipeline riêng vì code MobiVital chỉ chạy được LSTM — `mobivital_gen.py`
dòng 152 ghi cứng `LSTMMultiStep(...)`. Muốn thử TCN thì phải viết bộ chọn kênh
riêng, rồi chứng minh nó cho ra đúng kết quả code gốc.


## 0. Setup

In [ ]:
import os
import subprocess


def run(command):
    """Chạy một lệnh shell, in ra những gì nó in."""
    result = subprocess.run(command, shell=True, capture_output=True, text=True)
    print((result.stdout + result.stderr).strip())


REPO = "/content/UWB_RADAR"

if os.path.exists(REPO + "/.git"):
    os.chdir(REPO)
    run("git pull -q origin main")
else:
    os.chdir("/content")
    run("rm -rf " + REPO)
    run("git clone -q https://github.com/quangminhho004-blip/UWB_RADAR.git " + REPO)
    run("git clone -q https://github.com/nesl/mobivital-public.git "
        + REPO + "/external/mobivital")
    run("pip install -q einops")

os.chdir(REPO)
run("git rev-parse --short HEAD")
run("git -C external/mobivital rev-parse --short HEAD")

## 1. Dữ liệu — một bản CSV duy nhất

Giải nén thẳng vào thư mục MobiVital, đúng đường dẫn `prep_breath_final.py`
dòng 18 đòi. Không giữ bản thứ hai ở chỗ khác.

```
external/mobivital/dataset/mobivital/tripod/   1874 CSV   <- chỉ ở đây
data/                                          chỉ thứ pipeline của mình sinh ra
```

Hai pipeline đọc chung một bản CSV. Nhờ vậy khi mục 2 báo sai lệch bằng 0 thì
không ai cãi được là do hai bản dữ liệu khác nhau. Tải mất khoảng 3 phút.


In [ ]:
CSV_DIR = "external/mobivital/dataset/mobivital/tripod"

if os.path.exists(CSV_DIR):
    print("CSV đã có, bỏ qua")
else:
    run("apt-get install -qq -y aria2")
    run("aria2c -x16 -s16 -k5M --summary-interval=0 --console-log-level=error "
        "-d /content -o tripod.zip "
        "https://zenodo.org/api/records/15022885/files/tripod.zip/content")
    run("mkdir -p external/mobivital/dataset/mobivital")
    run("unzip -q -o /content/tripod.zip -d external/mobivital/dataset/mobivital/")

run("ls " + CSV_DIR + " | wc -l")


### Dọn chỗ cho pipeline gốc

Script chỉ thêm hai thứ MobiVital không có: thư mục lối tắt **vá 52 tên file lỗi
thời**, và `.git/info/exclude` để giấu dữ liệu khỏi git của họ.

Bảng kết quả họ commit sẵn ra đời trước khi dataset đổi tên lên Zenodo, 52 dòng
ghi mốc tháng 10 còn bản hiện tại là tháng 12. `evaluate.py` dòng 28 mở file
theo tên trong bảng, gặp 52 tên đó là chết giữa chừng. Chi tiết trong docstring
của script.


In [ ]:
!python scripts/mobivital/setup_dataset.py


### Dữ liệu cho pipeline gốc — chạy chính script của MobiVital

Lệnh đầu tiên trong README của họ, nguyên bản. Đọc CSV rồi ghi ra hai file:

```
CSV trong dataset/mobivital/tripod/
├─ lọc ABCDEFKL → data_final/training_breath_tripod_data.npy
└─ lọc GHIJ     → data_final/testing_breath_tripod_data.npy
```

Khoảng 10 phút.


In [ ]:
%cd /content/UWB_RADAR/external/mobivital
!python dataset_preparation/prep_breath_final.py
!ls -la data_final/


### Dữ liệu cho pipeline của mình

Đọc đúng bộ CSV đó, gom theo từng người:

```
CSV trong dataset/mobivital/tripod/
└─ lọc từng người → data/processed/by_user/A.npz … L.npz
```

Khoảng 20 phút.


In [ ]:
%cd /content/UWB_RADAR
if os.path.exists("data/processed/by_user/A.npz"):
    print("by_user đã có, bỏ qua")
else:
    run("python scripts/make_npz.py")

run("du -sh data/processed/*")


## 2. Đối chiếu hai bộ dữ liệu

Cùng CSV, hai đường đọc khác nhau. Phải ra đúng cái nhau:

```
A B C D E F K L  ->  by_user/{A..L}.npz  ==  training_breath_tripod_data.npy
G H I J          ->  by_user/{G..J}.npz  ==  testing_breath_tripod_data.npy
```

Khớp thì mọi thí nghiệm sau chỉ cần đọc `by_user/*.npz`, bỏ được CSV thô 13 GB.
Không khớp thì dừng, mọi so sánh về sau vô nghĩa.


In [ ]:
!python scripts/check_data.py


### Cửa sổ train cho pipeline của mình

`scripts/make_windows.py` cắt sẵn cửa sổ, hai bộ:

    dev_cv/       cắt riêng từng người A B C D E F K L, để ghép fold tuỳ ý
    final_train/  cắt gộp cả 8 người, đọc thẳng .npy vừa sinh ở trên nên thứ tự
                  buổi ghi giống hệt lúc MobiVital chạy

Khoảng 15 phút.


In [ ]:
if os.path.exists("data/processed/windows/final_train"):
    print("cửa sổ đã có, bỏ qua")
else:
    run("python scripts/make_windows.py")

run("ls data/processed/windows/dev_cv data/processed/windows/final_train")


## 3. TN0a — chấm bảng MobiVital commit sẵn

Lệnh của họ, README mục *Detailed Analysis*. Thêm cờ `-d` (cũng là cờ của họ) để
trỏ sang thư mục có 52 lối tắt tên cũ.

Bậc này **chưa đụng model** — bin và phép đã ghi sẵn trong bảng, bỏ model nào vào
cũng ra số đó. Giá trị của nó: ra khớp `0.819` trong bài báo nghĩa là CSV mình
tải về đúng bằng CSV họ dùng.


In [ ]:
%cd /content/UWB_RADAR/external/mobivital
!python -m inference.evaluate -m tripod_mobivital_pre_invert_0.9.txt \
        -d ./dataset/mobivital/tripod_old_names

## 4. TN0b — checkpoint của họ, tự sinh bảng

Lệnh của họ, README mục *Evaluating MobiVital*. Dùng checkpoint
`lstm_pred_tripod_0.9.pth` họ phát hành.

Với mỗi buổi ghi: dựng 240 ứng viên (120 kênh × 2 phép), lọc bằng
`invert_detector`, cắt 52 cửa sổ mỗi ứng viên, cho LSTM dự báo, chọn kênh có tổng
Pearson cao nhất. Bước chọn **không nhìn nhịp thở thật** — đó là điểm chính của
bài báo.


In [ ]:
!python -m inference.mobivital_gen

Đổi tên bảng vừa sinh. `mobivital_gen.py` dòng 121 luôn ghi ra đúng một tên nên
chạy TN0c sẽ đè mất — README của họ viết `-m YOUR_METHOD.txt` chính là ngụ ý việc
đổi tên này.

In [ ]:
!mv inference/methods/tripod_mobivital_pre_invert_0.9.txt inference/methods/TN0b.txt
!wc -l < inference/methods/TN0b.txt
!python -m inference.evaluate -m TN0b.txt --save_file scores_TN0b.csv

## 5. TN0c — train lại từ đầu

Lệnh của họ, README mục *Training*. Thêm `--model_name` (cờ của họ) để file `.pth`
mới không đè lên checkpoint gốc — chính README của họ cảnh báo:

> *Running the training script using the default parameters will override the
> checkpoint. So back the default checkpoint up if necessary.*

Cấu hình lấy từ `checkpoints/optimal_params.json` của họ: 20 epoch, Adam lr 1e-4,
batch 64, MSE. Khoảng **20 phút trên GPU**.


In [ ]:
!python -m training.autoreg_training --model_name lstm_retrained

In [ ]:
!python -m inference.mobivital_gen --model_name lstm_retrained
!mv inference/methods/tripod_mobivital_pre_invert_0.9.txt inference/methods/TN0c.txt
!python -m inference.evaluate -m TN0c.txt --save_file scores_TN0c.csv

Ba lần chấm dùng ba `--save_file` riêng. Lý do: `evaluate.py` dòng 67 gán Series
vào DataFrame đã có, pandas căn theo index cũ và **vứt key lạ** — dồn chung một
file thì những lần sau bị cắt mất dòng.

## 6. Gom kết quả và kiểm tra không đụng code họ


In [ ]:
%cd /content/UWB_RADAR
!cp external/mobivital/inference/methods/TN0b.txt        results/
!cp external/mobivital/inference/methods/TN0c.txt        results/
!cp external/mobivital/inference/methods/scores_TN0b.csv results/
!cp external/mobivital/inference/methods/scores_TN0c.csv results/
!cp external/mobivital/inference/methods/scores.csv      results/scores_TN0a.csv
!ls -la results/
print()
print("KIỂM TRA không sửa gì trong repo MobiVital:")
run("git -C external/mobivital status --short || echo '  git status trống'")

## 7. Pipeline của mình — làm lại đúng ba việc trên

Ba việc vừa rồi chạy bằng code MobiVital. Giờ làm lại y hệt bằng code của mình,
để đặt cạnh nhau mà so. Ba module đối một:

| việc | MobiVital | của mình |
|---|---|---|
| train | `training/autoreg_training.py` | `src/training.py` |
| chọn kênh | `inference/mobivital_gen.py` | `src/scoring.py` |
| chấm điểm, ghi bảng | `inference/evaluate.py` | `src/results.py` |

Mỗi bậc a → b → c thêm đúng một bộ phận của mình, nên bậc nào lệch đầu tiên thì
lỗi nằm ở đúng bộ phận vừa thêm:

    a   chỉ hàm chấm điểm
    b   thêm bộ chọn kênh
    c   thêm vòng train

a và b phải khớp tuyệt đối — cùng dữ liệu, cùng thuật toán, không có gì ngẫu
nhiên. c thì không, vì thứ tự xáo trộn dữ liệu khác nhau; lệch cỡ đổi seed là
bình thường.

Cần pipeline riêng vì `mobivital_gen.py` dòng 152 ghi cứng `LSTMMultiStep(...)` —
không nhét TCN vào được.


In [ ]:
%cd /content/UWB_RADAR
import csv
import numpy as np
import torch

from src import mobivital_reference as mv
from src import results
from src import scoring
from src import training

TEST_USERS = ["G", "H", "I", "J"]
device = "cuda" if torch.cuda.is_available() else "cpu"


def load_lstm(path):
    model = mv.new_lstm()
    model.load_state_dict(torch.load(path, map_location=device))
    return model.to(device).eval()


def save_and_report(name, rows):
    results.save_sessions("results/scores_" + name + ".csv", rows)
    micro = float(np.mean([row["pearson"] for row in rows]))
    print(len(rows), "buổi ghi   micro %.10f" % micro)
    return micro


print(device, torch.cuda.get_device_name(0) if device == "cuda" else "")


### 7a. Chấm bảng MobiVital commit sẵn

Đối chiếu với **mục 3**. Chỉ chạy hàm chấm điểm, chưa đụng model — bin và phép
đã ghi sẵn trong `results/TN0a.txt`.


In [ ]:
rows_a = scoring.score_from_txt("results/TN0a.txt", TEST_USERS)
ours_a = save_and_report("ours_a", rows_a)


### 7b. Checkpoint của MobiVital, bộ chọn kênh của mình

Đối chiếu với **mục 4**. Cùng file `.pth`, nhưng 240 ứng viên đi qua
`src/scoring.py` thay vì `inference/mobivital_gen.py`. Khoảng 10 phút.


In [ ]:
rows_b = scoring.score_all(TEST_USERS,
                           load_lstm("external/mobivital/checkpoints/lstm_pred_tripod_0.9.pth"))
scoring.write_txt(rows_b, "results/ours_b.txt")
ours_b = save_and_report("ours_b", rows_b)


### 7c. Train lại bằng vòng train của mình

Đối chiếu với **mục 5**. Cùng cấu hình MobiVital công bố trong
`checkpoints/optimal_params.json`: 20 epoch, Adam lr 1e-4, batch 64, MSE.

Cửa sổ đọc từ `data/processed/windows/final_train/` — cắt sẵn ở
`scripts/make_windows.py`, gộp cả 8 người A B C D E F K L đúng thứ tự MobiVital.
Khoảng 25 phút.


In [ ]:
X, y = training.load_windows(["train"], folder="data/processed/windows/final_train")
print(X.shape[0], "cửa sổ train")

training.set_seed(1234)
run = training.train(mv.new_lstm(), training.make_loader(X, y), None,
                     "runs/tn0/ours_c", loss_name="mse")
results.save_curve("runs/tn0/ours_c/curve.csv", run["curve"])


In [ ]:
rows_c = scoring.score_all(TEST_USERS, load_lstm(run["final_path"]))
scoring.write_txt(rows_c, "results/ours_c.txt")
ours_c = save_and_report("ours_c", rows_c)


## 8. Đối chiếu hai bảng

Hàng a và b khớp tới chữ số cuối chứng minh cùng lúc ba điều:

| | vì sao suy ra được |
|---|---|
| `by_user/*.npz` đúng bằng CSV gốc | `scoring.py` đọc `.npz`, `mobivital_gen.py` đọc CSV |
| bộ chọn kênh đúng | 537/537 buổi ghi chọn cùng bin, cùng phép |
| hàm chấm điểm đúng | 537 điểm lệch nhau 0 |


In [ ]:
def their_scores(path):
    """MobiVital ghi: cột 0 tên file, cột 1 điểm."""
    scores = {}
    for row in list(csv.reader(open(path)))[1:]:
        scores[row[0]] = float(row[1])
    return scores


def our_scores(path):
    scores = {}
    for row in csv.DictReader(open(path)):
        scores[row["session_file"]] = float(row["pearson"])
    return scores


def picks(path):
    """Bảng lựa chọn kênh: tên file -> (bin, phép)."""
    chosen = {}
    for row in csv.reader(open(path)):
        chosen[row[0]] = (row[1], row[2])
    return chosen


def compare(label, their_csv, our_csv, their_txt=None, our_txt=None):
    theirs = their_scores("results/" + their_csv)
    ours = our_scores("results/" + our_csv)
    gap = max(abs(theirs[f] - ours[f]) for f in theirs)

    same = ""
    if their_txt is not None:
        a = picks("results/" + their_txt)
        b = picks("results/" + our_txt)
        same = "%d/%d" % (sum(1 for f in a if a[f] == b.get(f)), len(a))

    print("%-24s %.10f   %.10f   %8.1e   %s"
          % (label, np.mean(list(theirs.values())), np.mean(list(ours.values())),
             gap, same))


print("%-24s %-14s %-14s %10s   %s"
      % ("", "MobiVital", "của mình", "lệch", "trùng lựa chọn"))
print("-" * 78)
compare("a  bảng có sẵn",       "scores_TN0a.csv", "scores_ours_a.csv")
compare("b  checkpoint có sẵn", "scores_TN0b.csv", "scores_ours_b.csv",
        "TN0b.txt", "ours_b.txt")
compare("c  train lại",         "scores_TN0c.csv", "scores_ours_c.csv",
        "TN0c.txt", "ours_c.txt")


## 9. Cất kết quả lên Drive

Giữ nguyên cấu trúc thư mục để bung ở máy vào đúng chỗ:

```bash
tar -xzf ~/Downloads/tn0.tar.gz -C /Users/udnb/Desktop/THESIS_GRADUATE/
```


In [ ]:
run("mkdir -p /content/drive/MyDrive/mobivital")
run("tar -czf /content/drive/MyDrive/mobivital/tn0.tar.gz results")
run("ls -la /content/drive/MyDrive/mobivital/tn0.tar.gz")

## Xong

`results/` giờ có hai bảng đối xứng:

```
                  lua chon kenh   diem tung buoi ghi
MobiVital  a      TN0a.txt        scores_TN0a.csv
           b      TN0b.txt        scores_TN0b.csv
           c      TN0c.txt        scores_TN0c.csv
minh       a      (dung TN0a)     scores_ours_a.csv
           b      ours_b.txt      scores_ours_b.csv
           c      ours_c.txt      scores_ours_c.csv
```

Từ đây thay LSTM bằng TCN, mọi khâu còn lại giữ nguyên. Mục 7 chính là thân của
hai script viết sau TN0 — chỉ thêm vòng fold và tham số dòng lệnh:

    scripts/run_cv.py           4 fold tren ABCDEFKL, chon cau hinh
    scripts/run_final_test.py   train du ABCDEFKL, test GHIJ mot lan duy nhat

Gọi từ Colab bằng đúng một dòng:

    !python scripts/run_cv.py --model ds_tcn --revin true
    !python scripts/run_final_test.py --model ds_tcn --revin true
